In [35]:
import json
import random
from pathlib import Path
from typing import Any
import dspy

from dspy.evaluate import Evaluate
from dspy.teleprompt import BootstrapFewShot, MIPROv2

import os
from dotenv import load_dotenv

# DATA_DIR= Path(__file__).parent/"data"
DATA_DIR = Path.cwd()/"data"
print(DATA_DIR)

d:\suvash\dspy-prac\data


In [15]:
def generat_qa_pairs(n:int=30, seed: int=42)->list[dict[str,Any]]:
    random.seed(seed)
    topics = [
        ("RAG", "Retrieval-Augmented Generation combines search with LLMs."),
        ("DSPy", "DSPy is a framework for programming language models."),
        ("Prompt Engineering", "Prompt engineering is the art of writing effective prompts."),
        ("Fine-tuning", "Fine-tuning adapts a pre-trained model to a specific task."),
        ("Vector DB", "Vector databases store embeddings for similarity search."),
        ("Chain-of-Thought", "CoT prompting asks the model to show its reasoning."),
        ("Attention", "Attention mechanisms let models focus on relevant tokens."),
        ("Transformers", "Transformers are neural networks based on self-attention."),
    ]
    qa_pairs =[]
    for i in range(n):
        topic, context = random.choice(topics)
        qa_pairs.append({
            "id":i,
            "question" : f"what is {topic}?",
            "context": context,
            "expected": context,
            "category": topic.upper().replace(" ", "_"),
            "difficulty": random.choice(['easy', "medium", "hard"])
        })

    return qa_pairs

In [23]:
qa_data=generat_qa_pairs(40)
for ques in qa_data:
    # print (ques)
    pass

In [25]:
examples = [
    dspy.Example(question=d["question"], answer=d["expected"]).with_inputs("question")
    for d in qa_data
]

# Split: 70% train, 30% dev

In [27]:
split=int(len(examples)*.7)
trainset=examples[:split]
devset=examples[split:]

In [28]:
print(f"Trainset: {len(trainset)} examples")
print(f"Devset: {len(devset)} examples")
print("\nSample train example:")
print(f"  Question: {trainset[0].question}")
print(f"  Answer: {trainset[0].answer}")

Trainset: 28 examples
Devset: 12 examples

Sample train example:
  Question: what is DSPy?
  Answer: DSPy is a framework for programming language models.


# Writing a custom Metric

In [29]:
def answer_metric(example, prediction, trace=None):
    """Check if predicted answer covers the expected answer's content

    Word-overlap ration (expected words found in prediction). A plain substring 
    check scores 0 on paraphrases, which leaves optimizers with nothing to bootstrap from.
    
    """
    expected = set(example.answer.lower().split())
    predicted = set(prediction.answer.lower().split())

    if not expected:
        return 0.0
    return 1.0 if len(expected & predicted)/ len(expected) >=0.5 else 0.0

In [30]:
ex = dspy.Example(question="What is RAG?", answer="Retrieval-Augmented Generation").with_inputs(
    "question"
)
pred = dspy.Prediction(answer="RAG stands for Retrieval-Augmented Generation")
print(f"Metric score: {answer_metric(ex, pred)}")

Metric score: 1.0


## Baseline Program

In [31]:
class QA(dspy.Signature):
    """Answer the questions with short factual responses."""
    question: str= dspy.InputField()
    answer:str=dspy.OutputField()

class SimpleQA(dspy.Module):
    def __init__(self):
        super().__init__()
        self.generate = dspy.ChainOfThought(QA)

    def forward(self, question):
        return self.generate(question=question)

baseline = SimpleQA()



## Evaluate the baseline

In [37]:
model_name = os.getenv("MODEL_NAME")
model_key=os.getenv("API_KEY")

In [38]:
kode_model = dspy.LM(
    f"openai/{model_name}", 
    api_key=model_key,
    base_url = "https://api.ai.kodekloud.com/v1"
)

# 2. Configure DSPy to use this model globally
dspy.configure(lm=kode_model )

In [40]:
evaluator= Evaluate(devset=devset, metric=answer_metric, num_threads=4, display_progress=True)
baseline_score = evaluator(baseline).score
print(f"\nBaseline score: {baseline_score:.2f}")

  0%|          | 0/12 [00:00<?, ?it/s]

Average Metric: 8.00 / 12 (66.7%): 100%|██████████| 12/12 [00:00<00:00, 34.08it/s]

2026/10/01 23:00:41 INFO dspy.evaluate.evaluate: Average Metric: 8.0 / 12 (66.7%)




Baseline score: 66.67


## BootstrapFeewShot - Quick Baseline Optimizer

In [41]:
# BootstrapFewShot: generates few-shot demos by running the program
teleprompter = BootstrapFewShot(metric=answer_metric, max_bootstrapped_demos=4)

optimized_bootstrap = teleprompter.compile(SimpleQA(), trainset=trainset)

# Evaluate optimized
bootstrap_score = evaluator(optimized_bootstrap).score
print(f"\nBootstrapFewShot score: {bootstrap_score:.2f}")
print(f"Improvement: {bootstrap_score - baseline_score:+.2f}")

 18%|█▊        | 5/28 [00:05<00:24,  1.06s/it]


Bootstrapped 4 full traces after 5 examples for up to 1 rounds, amounting to 5 attempts.
Average Metric: 10.00 / 12 (83.3%): 100%|██████████| 12/12 [00:04<00:00,  2.91it/s]

2026/10/01 23:02:35 INFO dspy.evaluate.evaluate: Average Metric: 10.0 / 12 (83.3%)




BootstrapFewShot score: 83.33
Improvement: +16.66


## MIPROv2 - Bayesian Optimization

In [44]:
# MIPROv2: Bayesian optimization of instructions + demos
# More expensive but generally better results

mipro = MIPROv2(
    metric=answer_metric,
    auto=None,  # dspy 3.3: opt out of auto budget to set candidates/trials manually
    num_candidates=5,  # Reduced for cost
    init_temperature=1.0,
)

optimized_mipro = mipro.compile(
    SimpleQA(),
    trainset=trainset,
    num_trials= 10,  # Configurable via .env os.getenv("DSPY_OPTIMIZER_TRIALS")
    valset=devset,
    minibatch=False,  # Small valset (12) — default minibatch size is 35
)

2026/10/01 23:10:55 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2026/10/01 23:10:55 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2026/10/01 23:10:55 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=5 sets of demonstrations...


Bootstrapping set 1/5
Bootstrapping set 2/5
Bootstrapping set 3/5


 18%|█▊        | 5/28 [00:00<00:03,  7.43it/s]


Bootstrapped 4 full traces after 5 examples for up to 1 rounds, amounting to 5 attempts.
Bootstrapping set 4/5


  4%|▎         | 1/28 [00:00<00:03,  7.93it/s]


...

2026/10/01 23:11:20 INFO dspy.evaluate.evaluate: Average Metric: 8.0 / 12 (66.7%)
2026/10/01 23:11:20 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 66.67 with parameters ['Predictor 0: Instruction 0', 'Predictor 0: Few-Shot Set 0'].
2026/10/01 23:11:20 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [66.67, 66.67, 66.67, 66.67, 58.33, 66.67, 83.33, 83.33, 66.67]
2026/10/01 23:11:20 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 83.33
2026/10/01 23:11:20 INFO dspy.teleprompt.mipro_optimizer_v2: ========================


2026/10/01 23:11:20 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Trial 10 / 10 =====



Average Metric: 8.00 / 12 (66.7%): 100%|██████████| 12/12 [00:03<00:00,  3.25it/s]

2026/10/01 23:11:24 INFO dspy.evaluate.evaluate: Average Metric: 8.0 / 12 (66.7%)
2026/10/01 23:11:24 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 66.67 with parameters ['Predictor 0: Instruction 3', 'Predictor 0: Few-Shot Set 1'].
2026/10/01 23:11:24 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [66.67, 66.67, 66.67, 66.67, 58.33, 66.67, 83.33, 83.33, 66.67, 66.67]
2026/10/01 23:11:24 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 83.33
2026/10/01 23:11:24 INFO dspy.teleprompt.mipro_optimizer_v2: =========================


2026/10/01 23:11:24 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Trial 11 / 10 =====



Average Metric: 8.00 / 12 (66.7%): 100%|██████████| 12/12 [00:03<00:00,  3.37it/s]

2026/10/01 23:11:27 INFO dspy.evaluate.evaluate: Average Metric: 8.0 / 12 (66.7%)
2026/10/01 23:11:27 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 66.67 with parameters ['Predictor 0: Instruction 0', 'Predictor 0: Few-Shot Set 2'].
2026/10/01 23:11:27 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [66.67, 66.67, 66.67, 66.67, 58.33, 66.67, 83.33, 83.33, 66.67, 66.67, 66.67]
2026/10/01 23:11:27 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 83.33
2026/10/01 23:11:27 INFO dspy.teleprompt.mipro_optimizer_v2: =========================


2026/10/01 23:11:27 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 83.33!


In [45]:
# Evaluate optimized
mipro_score = evaluator(optimized_mipro).score
print(f"\nMIPROv2 score: {mipro_score:.2f}")
print(f"Improvement over baseline: {mipro_score - baseline_score:+.2f}")
print(f"Improvement over Bootstrap: {mipro_score - bootstrap_score:+.2f}")

Average Metric: 10.00 / 12 (83.3%): 100%|██████████| 12/12 [00:00<00:00, 25.62it/s]

2026/10/01 23:11:54 INFO dspy.evaluate.evaluate: Average Metric: 10.0 / 12 (83.3%)




MIPROv2 score: 83.33
Improvement over baseline: +16.66
Improvement over Bootstrap: +0.00


## Inspecting Generated Instructions

In [46]:
# Inspect what the optimizer generated
print("Generated demos (BootstrapFewShot):")
demos = getattr(optimized_bootstrap.generate, "demos", [])  # absent if none kept
print(f"  {len(demos)} demos")
if not demos:
    print("  (No demos passed the metric — a looser metric yields more demos.)")
for i, demo in enumerate(demos[:2]):
    print(f"\n  Demo {i + 1}:")
    print(f"    Question: {demo.question}")
    print(f"    Answer: {demo.answer}")

print("\n" + "=" * 50)
print("Generated instructions may be embedded in the compiled program.")
print("Use optimized_mipro.save() to inspect the full program.")

Generated demos (BootstrapFewShot):
  0 demos
  (No demos passed the metric — a looser metric yields more demos.)

Generated instructions may be embedded in the compiled program.
Use optimized_mipro.save() to inspect the full program.


## Saving and Reloading

In [47]:
# Save optimized program
optimized_mipro.save("optimized_qa.json")
print("✅ Saved to optimized_qa.json")

# Reload
loaded = SimpleQA()
loaded.load("optimized_qa.json")

# Verify it works
result = loaded(question="What is DSPy?")
print(f"\nReloaded program answer: {result.answer}")

✅ Saved to optimized_qa.json

Reloaded program answer: DSPy is a framework for programming language models.
